# PRISM — Ollama + ngrok LLM Server

Run on Kaggle with **GPU** and **Internet** enabled.

**Secret required:** `NGROK_AUTH_TOKEN`

Copy the printed public URL into local `.env` as `LLM_BASE_URL`.

In [ ]:
import os, subprocess, time, urllib.request, json

NGROK_TOKEN = os.environ.get("NGROK_AUTH_TOKEN")
if not NGROK_TOKEN:
    raise ValueError("Set NGROK_AUTH_TOKEN in Kaggle Secrets")

MODEL = "llama3.2:3b"
OLLAMA_PORT = 11434

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start Ollama server in background
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("Ollama server starting…")

In [ ]:
# Pull model (small quantized model for Kaggle disk limits)
!ollama pull llama3.2:3b

In [ ]:
# Install and configure ngrok
!pip install -q pyngrok
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(OLLAMA_PORT, "http")
public_url = tunnel.public_url.rstrip("/")
print(f"\nPublic Ollama URL: {public_url}")
print(f"Set in local .env: LLM_BASE_URL={public_url}")

In [ ]:
# Verify Ollama is reachable via ngrok
import urllib.request, json

req = urllib.request.Request(f"{public_url}/api/tags")
req.add_header("ngrok-skip-browser-warning", "true")
with urllib.request.urlopen(req, timeout=30) as resp:
    tags = json.loads(resp.read())
print("Models available:", json.dumps(tags, indent=2))

In [ ]:
# Test OpenAI-compatible chat endpoint
import urllib.request, json

payload = json.dumps({
    "model": "llama3.2",
    "messages": [{"role": "user", "content": "Say hello in one sentence."}],
    "stream": False
}).encode()
req = urllib.request.Request(
    f"{public_url}/v1/chat/completions",
    data=payload,
    headers={"Content-Type": "application/json", "ngrok-skip-browser-warning": "true"},
    method="POST"
)
with urllib.request.urlopen(req, timeout=120) as resp:
    result = json.loads(resp.read())
print(result["choices"][0]["message"]["content"])

## Keep this notebook running

The ngrok tunnel stays active while this notebook session is alive. If Kaggle times out, re-run and update `LLM_BASE_URL` in your local `.env`.